# 06. Laya 체험 — 다국어 System 1 decision engine

[Laya](https://github.com/NandhaKishorM/laya)는 Convai Innovations가 공개한 오픈소스(Apache-2.0) decision model 패밀리입니다.

- **non-autoregressive**: BERT 계열 encoder(English 421M / multilingual 322M)가 한 번의 forward pass로 choice/noul/score에 답합니다 — 지금까지 본 decision model 중 가장 작습니다 (CPU로도 실용적).
- **다국어 100+**: `Router()`가 입력 언어를 감지해 한국어 등 비영어 입력을 `laya-multilingual` checkpoint로 자동 라우팅합니다.
- RLCD(strictly proper scoring rule 기반 RL)로 학습된 **calibrated 확률** 반환.
- `laya[serve]`는 **`/v1/systemone`** (Jev 호환 wire format)도 제공하고, LangChain/LangGraph/LlamaIndex/CrewAI/MCP 통합, TypeScript 포트(`npm install laya-ts`)까지 있습니다.
- 완전 무료 self-host — API key/계정 불필요.

이 노트북은 HTTP 서버 없이 **Python native `Router` API를 kernel 안에서 직접** 사용합니다 (kernel 환경에 `pip install laya` 필요 — workshop kernel에는 사전 설치되어 있으며, checkpoint는 HF_HOME 캐시에 다운로드됩니다).

In [1]:
import sys, os, time
from pathlib import Path; sys.path.insert(0, str(Path.cwd().parent))  # ws_common 경로
os.environ.setdefault("HF_HOME", os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface")))
import ws_common as W
from laya import Router

router = Router()    # 첫 사용 시 checkpoint 다운로드
print("laya Router ready")

laya Router ready


## 1. 한국어 질문 하나로 판단 3종을 한 번에

Laya는 한 state에 여러 질문을 묶어 보낼 수 있습니다. 금융 상담 문맥으로 **업무 분류(choice) + 긴급성(noul) + 고객 감정(score)** 을 동시에 물어봅니다. `routing.model`이 `multilingual`로 찍히는지(한국어 자동 감지) 확인하세요.

In [2]:
state = "지갑을 도둑맞았어요! 카드 정지시켜 주세요. 빨리요!!"
questions = {
    "d2_team": {"type": "choice",
                "instructions": "Which banking support specialist team should handle this customer message?",
                "criteria": {c: None for c in
                             ["cards", "card_security", "disputes_fees", "transfers",
                              "accounts_balance", "top_up", "exchange_fx", "refunds",
                              "declined_pay", "identity_verif", "account_mgmt", "general_faq"]}},
    "urgent": {"type": "noul", "instructions": "Does this message convey urgency?"},
    "emotion": {"type": "score", "instructions": "How upset is the customer?",
                "criteria": ["calm", "worried", "very upset"]},
}

router.predict(state, questions)   # warm-up (측정 제외)
t0 = time.perf_counter()
r = router.predict(state, questions)
ms = (time.perf_counter() - t0) * 1000

print(f"checkpoint: {r['routing']['model']}   (질문 3개 합산 {ms:,.1f}ms)")
a = r["answers"]
print(f"  d2_team -> {a['d2_team']['choice']} (confidence {a['d2_team']['confidence']:.3f})")
print(f"  urgent  -> P(yes) = {a['urgent']['noul']:.3f}")
print(f"  emotion -> score {a['emotion']['score']:.2f}  (0=calm ~ 2=very upset)")

/data/venvs/wsverify/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:  20%|██        | 1/5 [00:00<00:02,  1.96it/s]

Fetching 5 files:  80%|████████  | 4/5 [00:02<00:00,  1.88it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:07<00:00,  1.93s/it]

Fetching 5 files: 100%|██████████| 5/5 [00:07<00:00,  1.53s/it]

checkpoint: multilingual   (질문 3개 합산 193.8ms)
  d2_team -> card_security (confidence 0.761)
  urgent  -> P(yes) = 0.957
  emotion -> score 1.50  (0=calm ~ 2=very upset)


## 2. 우리 LangGraph workflow에 꽂기 — adapter 20줄

Laya는 Python 객체 API이므로, `.decide()` 서명에 맞춘 **얇은 adapter**만 쓰면 지금까지의 graph에 그대로 들어갑니다. backend 교체가 얼마나 작은 일인지 보여주는 예입니다.
> adapter에서 `model="multilingual"`을 **고정**합니다. 자동 라우팅에 맡기면 영어 섞인 긴 state(tool JSON 등)가 english checkpoint로 가면서 체크포인트 전환이 반복돼 느려지고, 긴 입력은 `max_len=8192`가 필요합니다. 또한 Laya는 **선택지 11개 이상 choice의 confidence는 uncalibrated**라고 스스로 경고합니다(D2 12지 해당) — label은 쓰되 confidence 기반 설계는 피하세요.


In [3]:
from agent.decisions import DecisionResult, _fallback

class LayaDecision:
    name = "laya"
    def __init__(self, router):
        self.router = router
    def decide(self, *, state_text, question, choices, decision_id):
        qs = {decision_id: {"type": "choice", "instructions": question,
                            "criteria": {c: None for c in choices}}}
        t0 = time.perf_counter()
        r = self.router.predict(state_text, qs, model="multilingual", max_len=8192)
        ms = (time.perf_counter() - t0) * 1000
        ans = r["answers"][decision_id]
        label, valid = ans["choice"], ans["choice"] in choices
        if not valid:
            label = _fallback(decision_id, choices)
        return DecisionResult(label=label, confidence=ans.get("confidence"),
                              latency_ms=round(ms, 2), valid=valid,
                              usage={}, cost_usd=0.0, raw=str(ans)[:200])

laya_dec = LayaDecision(router)
print("adapter ready")

adapter ready


In [4]:
from agent.graph_v2 import build_graph_v2
from agent.decisions import SystemOneDecision

client, llm = W.make_clients()
graph_laya    = build_graph_v2(laya_dec, llm)
graph_strands = build_graph_v2(SystemOneDecision(), llm)
sc = W.SCENARIOS[1]
print("시나리오:", sc["title"], "|", sc["question"])

시나리오: 모호한 문의 -> clarification loop | 카드가 안 긁히는데 어떻게 해야 하나요?


In [5]:
res_laya = W.run_verbose(graph_laya, sc["question"], sc["script"], "Laya")


[Laya] 고객: 카드가 안 긁히는데 어떻게 해야 하나요?

STEP 1 ▶ guardrail_check — D1 접수 심사: 정상 문의 / 범위 밖 / compliance 위반
      판정[D1] -> in_scope   (103ms, confidence 0.177)

STEP 2 ▶ intent_router — D2 업무 분류: 12개 전문 상담 category 중 선택
      판정[D2] -> account_mgmt   (80ms, confidence 0.614)



STEP 3 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> need_clarification   (100ms, confidence 0.795)



STEP 4 ▶ ask_clarify — 되묻기: Sonnet이 질문 생성 -> 고객 답변 수신
      생성(sonnet-4-5)   (2,084ms)
      상담 agent: 어떤 카드가 안 긁히시나요? (체크카드, 신용카드 등 카드 종류와 카드 뒷면 마지막 4자리를 알려주세요)
      고객: 어제부터 오프라인 매장 단말기에서만 승인이 거절돼요. 관련 정보: Visa 체크카드(끝자리 7712), 마지막 시도 2026-10-27 TechMart 강남점, 금액 45,000원입니다.

STEP 5 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> info_sufficient   (192ms, confidence 0.014)



STEP 6 ▶ tool_step — D3 tool 선택 -> tool 실행 -> D6 결과 검토(성공/재시도/포기)
      판정[D3] -> account_policy_info   (161ms, confidence 0.795)
      tool 실행: account_policy_info   (0.0ms)
      판정[D6] -> goal_met   (166ms, confidence 0.689)



STEP 7 ▶ summarize — Sonnet이 tool 결과를 고객 응대용 사실 요약으로 정리
      생성(sonnet-4-5)   (2,490ms)
      요약(첫 줄): Based on the tool result, here are the key resolution facts:



STEP 8 ▶ risk_check — D7 리스크 점검: fraud/AML 모니터링 flag 유무
      판정[D7] -> needs_verification   (224ms, confidence 0.000)

STEP 9 ▶ kyc_verify — KYC 재검증 tool 실행 -> D8 통과 여부 판정
      tool 실행: run_kyc_check   (0.0ms)
      판정[D8] -> verified   (109ms, confidence 0.610)



STEP 10 ▶ escalation_gate — D4 처리 방침: 종결 / 추가 질문 / 상담사 이관
      판정[D4] -> clarify   (407ms, confidence 0.755)



STEP 11 ▶ respond — Sonnet이 최종 고객 응답 작성
      생성(sonnet-4-5)   (3,763ms)
      --- 최종 응답 ---
      고객님, 정보 감사합니다.
      
      Visa 체크카드(끝자리 7712) 관련하여 몇 가지 추가 확인이 필요합니다:
      
      1. **온라인 결제**는 정상적으로 되시나요?
      2. 다른 오프라인 매장에서도 시도해보셨나요?
      3. 카드 **자기띠(검은 줄) 부분**에 손상이나 긁힌 자국이 있나요?
      4. 최근 **해외 거래**나 **고액 결제** 시도가 있으셨나요?
      
      위 정보를 알려주시면 정확한 원인 파악 후 즉시 해결 방법을 안내해드리겠습니다.

>>> [Laya] 결정 9회 합계 1,542ms | 생성 8,337ms | e2e 9,891ms


In [6]:
res_strands = W.run_verbose(graph_strands, sc["question"], sc["script"], "StrandsDecider")


[StrandsDecider] 고객: 카드가 안 긁히는데 어떻게 해야 하나요?

STEP 1 ▶ guardrail_check — D1 접수 심사: 정상 문의 / 범위 밖 / compliance 위반
      판정[D1] -> in_scope   (124ms, confidence 0.907)



STEP 2 ▶ intent_router — D2 업무 분류: 12개 전문 상담 category 중 선택
      판정[D2] -> declined_pay   (104ms, confidence 0.466)

STEP 3 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> need_clarification   (105ms, confidence 0.761)



STEP 4 ▶ ask_clarify — 되묻기: Sonnet이 질문 생성 -> 고객 답변 수신
      생성(sonnet-4-5)   (1,670ms)
      상담 agent: 어떤 카드가 안 긁히시나요? 신용카드인지 체크카드인지, 그리고 카드 뒷면 마지막 4자리 번호를 알려주시겠어요?
      고객: 어제부터 오프라인 매장 단말기에서만 승인이 거절돼요. 관련 정보: Visa 체크카드(끝자리 7712), 마지막 시도 2026-10-27 TechMart 강남점, 금액 45,000원입니다.

STEP 5 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> info_sufficient   (153ms, confidence 0.580)



STEP 6 ▶ tool_step — D3 tool 선택 -> tool 실행 -> D6 결과 검토(성공/재시도/포기)
      판정[D3] -> check_payment_status   (153ms, confidence 0.042)
      tool 실행: check_payment_status   (0.0ms)
      판정[D6] -> goal_met   (107ms, confidence 0.537)



STEP 7 ▶ summarize — Sonnet이 tool 결과를 고객 응대용 사실 요약으로 정리
      생성(sonnet-4-5)   (3,345ms)
      요약(첫 줄): Based on the tool result, here are the key resolution facts:

STEP 8 ▶ risk_check — D7 리스크 점검: fraud/AML 모니터링 flag 유무
      판정[D7] -> no_risk   (157ms, confidence 0.841)



STEP 9 ▶ escalation_gate — D4 처리 방침: 종결 / 추가 질문 / 상담사 이관
      판정[D4] -> clarify   (156ms, confidence 0.596)



STEP 10 ▶ respond — Sonnet이 최종 고객 응답 작성
      생성(sonnet-4-5)   (4,018ms)
      --- 최종 응답 ---
      고객님, 시스템 점검 결과 Visa 체크카드(끝자리 7712)는 정상 상태입니다.
      
      오프라인 매장에서만 승인 거절이 발생하는 경우, 몇 가지 확인이 필요합니다:
      
      1. **계좌 잔액**이 충분하신가요?
      2. **일일 이용한도**를 초과하지 않으셨나요?
      3. 다른 오프라인 매장에서도 같은 문제가 발생하나요?
      
      위 사항들을 확인해 주시고, 잔액과 한도에 문제가 없다면 카드 재발급이 필요할 수 있습니다. 
      
      추가로 필요하신 정보를 알려주시면 더 정확한 안내를 도와드리겠습니다.
      
      (참조번호: REF-ws-StrandsDecider-285852)

>>> [StrandsDecider] 결정 8회 합계 1,059ms | 생성 9,033ms | e2e 10,101ms


In [7]:
W.compare(res_laya, res_strands)

| 지표 | Laya | StrandsDecider | 배수(Laya/StrandsDecider) |
|---|---:|---:|---:|
| 결정 횟수 | 9 | 8 | |
| **결정 latency 합계** | **1,542ms** | **1,059ms** | **1.5x** |
| 생성(LLM) latency | 8,337ms | 9,033ms | **0.9x** |
| e2e | 9,891ms | 10,101ms | **1.0x** |

**결정 계층 속도 차이: 1.5x** (e2e 단축 -210ms)

**결정별 latency**

| 노드 | 결정 | Laya | StrandsDecider | 배수 | 판정 label |
|---|---|---:|---:|---:|---|
| guardrail_check | D1 | 103ms | 124ms | **0.8x** | in_scope (동일) |
| intent_router | D2 | 80ms | 104ms | **0.8x** | Laya=account_mgmt / StrandsDecider=declined_pay |
| clarify_gate | D5 | 100ms | 105ms | **1.0x** | need_clarification (동일) |
| clarify_gate | D5 | 192ms | 153ms | **1.3x** | info_sufficient (동일) |
| tool_select | D3 | 161ms | 153ms | **1.0x** | Laya=account_policy_info / StrandsDecider=check_payment_status |
| tool_review | D6 | 166ms | 107ms | **1.6x** | goal_met (동일) |
| risk_check | D7 | 224ms | 157ms | **1.4x** | Laya=needs_verification / StrandsDecider=no_risk |
| kyc_review | D8 | 109ms | 156ms | **0.7x** | Laya=verified / StrandsDecider=clarify |

## 한국어 판단 품질 실측 (backend 비교)

이 워크샵의 한국어 160건(객관적 gold)으로 D1/D2를 전수 측정한 결과입니다 (companion decider-quality bench):

| backend | D1 guardrail (3지) | D2 routing strict/lenient (12지) | p50 latency |
|---|---|---|---|
| **strands-decider E4B** (이 워크샵의 backend) | **0.938** | **0.636 / 0.729** | ~104ms |
| 범용 8B VLM (참고) | 0.913 | 0.629 / 0.707 | ~33ms |
| strands-decider 2B | 0.613 | 0.500 / 0.593 | ~85ms |
| Laya multilingual 322M | 0.419 | 0.357 / 0.379 | ~21ms |

**해석**: 판단 태스크에 적응(adapt)된 **전용 decision model(E4B)이 두 배 큰 범용 8B VLM을 품질에서 앞섭니다** — 전용 적응이 범용 용량을 이긴 결과입니다. 같은 전용 계열 안에서는 용량이 여전히 상한을 결정합니다 (E4B > 2B). Laya 측정치는 공식 문서의 zero-shot 수치(0.362)와 일치하며, Laya는 RLCD fine-tuning으로 0.766까지 오르는 **'튜닝 전제' 모델**입니다. 반대로 latency·풋프린트는 작을수록 유리합니다 — 품질/비용/속도의 3축 trade-off를 그대로 보여주는 결과입니다.


## 3. 정리 — decision model 선택지 지도

이 폴더(30_other_deciders)까지 포함하면, 같은 금융 상담 판단을 네 가지 방식으로 체험했습니다.

| 방식 | 판단 주체 | 판정 latency(이 환경) | 비용 구조 | 특징 |
|---|---|---|---|---|
| Bedrock Haiku 4.5 | 범용 LLM 생성 | 수백 ms | 토큰 과금 | 품질 기준선, 관리형 |
| strands-decider E4B /v1/systemone (01-04) | 전용 decision model (GPU) | **~100ms** | self-host (GPU 재사용) | **calibrated confidence**, 한국어 판단 품질 1위, Strands SDK 통합 |
| Bedrock Intelligent Routing (05) | 판단이 아니라 **생성 모델 선택** | 선택된 모델의 생성 시간 | $1/1,000건 + 토큰 | 생성 비용 최적화 |
| Laya 322-421M (06) | 전용 decision model | **~30ms** (in-process) | self-host (CPU 가능) | 다국어 자동 라우팅, 최소 풋프린트 |

- **한국어 관점**: Laya는 multilingual checkpoint가 자동 선택되어 한국어를 1급으로 다루지만, zero-shot 품질은 위 bench처럼 낮습니다 — 반면 strands-decider E4B는 fine-tuning 없이도 한국어 판단 품질 1위입니다.
- **주의**: confidence의 의미(캘리브레이션 방식)가 vendor마다 다릅니다. threshold는 제품 간 이식되지 않으므로, 반드시 자기 데이터로 reliability를 다시 측정하세요.
- zero-shot으로 부족하면 Laya는 RLCD fine-tuning 경로(공식 노트북)도 제공합니다 — typed-decisions benchmark에서 base 0.362 -> fine-tuned 0.766.
- 다음: `[KR]_07_cost_routing.ipynb` — decision layer로 생성 트래픽의 비용을 줄이는 난이도 라우팅.
